<!-- Architecture
Image Input → LangChain → [Gemini / HuggingFace / OpenRouter / OpenAI] → Text Output -->

In [ ]:
                                    #LangChain + HuggingFace Inference Providers

# Uses ChatOpenAI pointed at HF's OpenAI-compatible router (the only reliable LangChain path)
# Models: aya-vision-32b (cohere)

#Image to Text 
import os
import base64
from dotenv import load_dotenv
from pydantic import SecretStr
from langchain_openai import ChatOpenAI          # pip install langchain-openai
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

# ─────────────────────────────────────────────
# Confirmed FREE vision models for HF Pro users
# Format for HF router: "model_id:provider"
# ─────────────────────────────────────────────
MODELS = {
    "aya": {
        "model":    "CohereLabs/aya-vision-32b",
        "provider": "cohere",
        "router_model": "CohereLabs/aya-vision-32b:cohere",
    },
}

TEST_IMAGE_PATH = "C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Assignment\Access_Different_LLMs\images\AIImage.jpg"


# ─────────────────────────────────────────────
# Helper: local path → base64 data URI
# ─────────────────────────────────────────────
def load_image(image_source: str) -> str:
    """Convert local file to base64 data URI. or pass image as URL directly."""

    if not os.path.exists(image_source):
        raise FileNotFoundError(f"Image not found: {image_source}")

    with open(image_source, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    ext = image_source.rsplit(".", 1)[-1].lower()
    mime = {"jpg": "image/jpeg", "jpeg": "image/jpeg",
            "png": "image/png",  "gif": "image/gif",
            "webp": "image/webp"}.get(ext, "image/jpeg")

    return f"data:{mime};base64,{encoded}"


# ─────────────────────────────────────────────
# Chain builder — uses HF's OpenAI-compatible router
# LangChain's HuggingFaceEndpoint does NOT support
# the new Inference Providers system, so we use
# ChatOpenAI pointed at https://router.huggingface.co/v1
# ─────────────────────────────────────────────
def create_vision_chain(router_model: str):
    llm = ChatOpenAI(
        model=router_model,
        base_url="https://router.huggingface.co/v1",    # HF's OpenAI-compatible router
        api_key=SecretStr(HF_TOKEN),
        max_tokens=512,
    )
    return llm | StrOutputParser()


# ─────────────────────────────────────────────
# Main inference function
# ─────────────────────────────────────────────
def generate_text_from_image(
    image_source: str,
    prompt: str = "Describe this image in detail.",
    model_key: str = "aya",          # "qwen" or "aya"
) -> str:
    config = MODELS[model_key]
    print(f"\n{'='*50}")
    print(f"Model    : {config['model']}")
    print(f"Provider : {config['provider']}")
    print(f"Image    : {image_source}")
    print(f"Prompt   : {prompt}")
    print(f"{'='*50}\n")

    image_url = load_image(image_source)

    message = HumanMessage(
        content=[
            {"type": "image_url", "image_url": {"url": image_url}},
            {"type": "text", "text": prompt},
        ]
    )

    chain = create_vision_chain(config["router_model"])
    return chain.invoke([message])


# ─────────────────────────────────────────────
# Run LLM
# ─────────────────────────────────────────────
if __name__ == "__main__":
    PROMPT = "Describe this image in detail."

    # ── Model 2: aya-vision-32b (cohere) ──
    print(">>> Running Model 2: aya-vision-32b (cohere)")
    try:
        output2 = generate_text_from_image(
            image_source=TEST_IMAGE_PATH,
            prompt=PROMPT,
            model_key="aya",
        )
        print("--- Aya Vision Output ---")
        print(output2)
    except Exception as e:
        print(f"[Aya Error] {e}")

>>> Running Model 2: aya-vision-32b (cohere)

Model    : CohereLabs/aya-vision-32b
Provider : cohere
Image    : C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Assignment\Access_Different_LLMs\images\AIImage.jpg
Prompt   : Describe this image in detail.

--- Aya Vision Output ---
This image is a vibrant and complex digital illustration that captures the essence of artificial intelligence (AI) and its integration into various aspects of modern life. At the center of the composition is a stylized representation of a human brain, which serves as a metaphor for AI's cognitive abilities. The brain is depicted in a gradient of blue hues, with intricate neural pathways and synapses highlighted in white, giving it a futuristic and technological appearance.

Surrounding the brain are numerous icons and symbols that represent different industries and applications where AI is making an impact. These include:

1. **Technology**: Icons of a laptop, smartphone, and a circuit board sign

In [ ]:
                            #LangChain + Google Gemini Providers

#Image to Text
import os
import base64
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import mimetypes
from pathlib import Path


load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


def is_url(source: str) -> bool:
    """Return True if the source is an HTTP/HTTPS URL."""
    return source.startswith("http://") or source.startswith("https://")

if not GOOGLE_API_KEY:
    raise EnvironmentError(
        "GOOGLE_API_KEY is not set.\n"
        "Add it to your .env file:  GOOGLE_API_KEY=your_key_here\n"
        "Get a key at: https://aistudio.google.com/app/apikey"
    )
#  Actually use ChatGoogleGenerativeAI and StrOutputParser
def create_gemini_chain(model: str = "ggemini-2.5-flash"):
    """Build a LangChain chain: Gemini LLM → StrOutputParser."""
    llm = ChatGoogleGenerativeAI(
        model=model,
        google_api_key=GOOGLE_API_KEY,
    )
    return llm | StrOutputParser()

def encode_image_to_base64(file_path: str) -> tuple[str, str]:
    """
    Read a local image file and encode it to base64.

    Args:
        file_path: Path to a local image file (.jpg, .png, .webp, etc.)

    Returns:
        Tuple of (base64_string, mime_type)
        e.g. ("iVBORw0KGgo...", "image/png")
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"Image not found: {file_path}")

    # Detect MIME type from file extension
    mime_type, _ = mimetypes.guess_type(str(path))
    if not mime_type:
        mime_type = "image/jpeg"  # fallback default

    with open(path, "rb") as f:
        b64_string = base64.b64encode(f.read()).decode("utf-8")

    return b64_string, mime_type

def build_multimodal_message(image_source: str, prompt: str) -> HumanMessage:
    """
    Build a LangChain HumanMessage with image + text content.

    Handles two input types automatically:
      - URL   → image passed directly as image_url
      - File  → image read, base64-encoded, passed as data URL

    Args:
        image_source: Public image URL or local file path.
        prompt      : Text instruction to send with the image.

    Returns:
        HumanMessage with multimodal content list.

    Example:
        >>> msg = build_multimodal_message("./photo.jpg", "Describe this image.")
        >>> msg = build_multimodal_message("https://example.com/img.jpg", "What is in this image?")
    """

    if is_url(image_source):
        # ── URL-based image ──────────────────────────────────────────────────
        # Gemini and OpenAI accept public URLs directly.
        image_block = {
            "type": "image_url",
            "image_url": {"url": image_source},
        }

    else:
        # ── Local file: encode to base64 data URL ────────────────────────────
        b64_string, mime_type = encode_image_to_base64(image_source)

        # data URL format: "data:<mime_type>;base64,<encoded_data>"
        data_url = f"data:{mime_type};base64,{b64_string}"

        image_block = {
            "type": "image_url",
            "image_url": {"url": data_url},
        }

    # ── Text prompt block ────────────────────────────────────────────────────
    text_block = {
        "type": "text",
        "text": prompt,
    }

    # ── Combine into a single HumanMessage ───────────────────────────────────
    # LangChain's content list format: [image_block, text_block]
    return HumanMessage(content=[image_block, text_block])


def generate_text_from_image(
    image_source: str,
    prompt: str = "Describe this image in detail.",
    model: str = "gemini-2.5-flash",
) -> str:
    """Send image + prompt to Gemini and return the text response."""
    print(f"\n[Gemini] Model  : {model}")
    print(f"[Gemini] Image  : {image_source}")
    print(f"[Gemini] Prompt : {prompt}\n")

    message = build_multimodal_message(image_source, prompt)
    chain = create_gemini_chain(model=model)
    return chain.invoke([message])

if __name__ == "__main__":
    # TEST_URL = (
    #     "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/"
    #     "PNG_transparency_demonstration_1.png/"
    #     "280px-PNG_transparency_demonstration_1.png"
    # )

    TEST_URL=TEST_IMAGE_PATH = "C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Assignment\Access_Different_LLMs\images\AIImage.jpg"

    # ✅ Fix 4: Actually invoke the chain and print the result
    try:
        output = generate_text_from_image(
            image_source=TEST_URL,
            prompt="Describe this image in detail.",
        )
        print("--- Gemini Output ---")
        print(output)
    except Exception as e:
        print(f"[Error] {e}")



[Gemini] Model  : gemini-2.5-flash
[Gemini] Image  : C:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\Assignment\Access_Different_LLMs\images\AIImage.jpg
[Gemini] Prompt : Describe this image in detail.



--- Gemini Output ---
This image is a digital graphic illustrating the concept and widespread integration of Artificial Intelligence (AI).

Here's a detailed description:

**Central Elements:**
*   **"AI" Text:** Dominating the center of the image are the large, bold, white capital letters "AI" (for Artificial Intelligence).
*   **Neural Network/Brain Motif:** Surrounding the "AI" text is a glowing, abstract network of interconnected lines and nodes, strongly resembling a human brain or a neural network. This network is rendered in shades of light blue and white, giving it a luminous, technological appearance.

**Radiating Icons:**
*   Connected to and radiating outwards from this central brain-like network is a multitude of small, white, outline icons. These icons represent various aspects of modern life, technology, and industry, signifying AI's pervasive influence.
*   **Examples of Icons (clockwise from top left):**
    *   Shopping cart
    *   Skyscraper/building
    *   Bowl/dis